In [1]:
import argparse
import csv
from datetime import datetime
from pathlib import Path

import yt_dlp


def format_upload_date(value):
    """Convert YYYYMMDD to YYYY-MM-DD."""
    if not value:
        return None

    try:
        return datetime.strptime(value, "%Y%m%d").date().isoformat()
    except ValueError:
        return value


def download_channel_metadata(channel_url, output_file):
    options = {
        "skip_download": True,
        "ignoreerrors": True,
        "quiet": True,

        # Do not enable extract_flat: upload dates and descriptions may
        # otherwise be missing because individual video pages aren't examined.
        "extract_flat": False,

        # Be a little gentler when processing large channels.
        "sleep_interval": 1,
        "max_sleep_interval": 3,
    }

    rows = []

    with yt_dlp.YoutubeDL(options) as ydl:
        channel = ydl.extract_info(channel_url, download=False)

        if not channel:
            raise RuntimeError("Could not retrieve the channel.")

        entries = channel.get("entries") or []

        for video in entries:
            if not video:
                continue  # Deleted, private, or otherwise unavailable video

            rows.append(
                {
                    "title": video.get("title"),
                    "description": video.get("description"),
                    "upload_date": format_upload_date(
                        video.get("upload_date")
                    ),
                    "views": video.get("view_count"),
                    "url": video.get("webpage_url")
                    or f"https://www.youtube.com/watch?v={video.get('id')}",
                }
            )

    output_path = Path(output_file)

    with output_path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "title",
                "description",
                "upload_date",
                "views",
                "url",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved metadata for {len(rows)} videos to {output_path}")

download_channel_metadata("https://www.youtube.com/@scads-ai", "videos.csv")

Saved metadata for 76 videos to videos.csv
